# Step 6: Fine-tune the Transformer Baseline
Using HuggingFace Trainer to fine-tune MuRIL on the cleaned splits, with class-weighted loss and evaluating on Macro-F1.


In [1]:
import torch
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA runtime:", torch.version.cuda)

Python: c:\Users\Jatin\anaconda3\envs\torch_gpu\python.exe
PyTorch: 2.6.0+cu124
CUDA: True
CUDA runtime: 12.4


In [2]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [ ]:
# Load datasets
train_df = pd.read_csv("data/processed/train.csv").dropna(subset=['text'])
val_df = pd.read_csv("data/processed/validation.csv").dropna(subset=['text'])
test_df = pd.read_csv("data/processed/test.csv").dropna(subset=['text'])
    

: 

In [ ]:
# Model prep
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Based on Step 5, set MAX_LEN appropriately (e.g. 256)
MAX_LEN = 256 

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    return {"accuracy": acc, "f1_macro": f1}


In [ ]:
# Handle Class Imbalance with Custom Trainer
# Calculate class weights (inverse of frequency)
class_weights = (1 - (train_df['label'].value_counts().sort_index() / len(train_df))).values
class_weights = torch.tensor(class_weights, dtype=torch.float32).to('cuda' if torch.cuda.is_available() else 'cpu')

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # evaluate each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir='./logs',
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)


In [ ]:
# Train
print("Starting Training...")
trainer.train()


In [ ]:
# Save Best Model
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")
print("Best model saved to ./best_model")
